# Modelos de Clasificacion: Arboles y Ensambles

**Diplomado en Machine Learning — Curso III, Bloque 1**

---

En este notebook vamos a recorrer, paso a paso, los modelos de clasificacion basados en arboles:

1. **Arboles de decision** — como funcionan por dentro, con un ejemplo diminuto.
2. **Random Forest** — que pasa cuando juntamos muchos arboles.
3. **Gradient Boosting y XGBoost** — arboles que aprenden de sus errores.
4. **Comparacion practica** — los tres modelos frente a frente con datos reales.
5. **Hiperparametros** — como encontrar la mejor configuracion con GridSearch y RandomizedSearch.

Todo esta pensado para que se entienda *antes* de memorizar. Primero usamos datos pequenos ("de juguete") para ver que ocurre detras de camaras, y despues pasamos a un dataset real de sklearn convertido a pandas, tal como harias con un CSV o Excel propio.

> **Requisitos:** Python 3.8+, scikit-learn, pandas, matplotlib, seaborn, xgboost.
> Este notebook esta listo para ejecutarse en **Google Colab** sin cambios.


---
## 0. Instalacion y preparacion del entorno

Instalamos las librerias necesarias. En Google Colab la mayoria ya estan disponibles; solo necesitamos asegurar xgboost.


In [ ]:
# Instalacion (solo xgboost suele faltar en Colab)
!pip install xgboost -q

In [ ]:
# Importaciones generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Configuracion visual
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Para reproducibilidad
SEED = 42
np.random.seed(SEED)

print("Todo listo.")

---
---
# PARTE 1: Arboles de Decision

## 1.1 La idea central

Un arbol de decision funciona exactamente como el juego de las 20 preguntas:

- Hace una pregunta sobre los datos.
- Segun la respuesta (si/no), divide los datos en dos grupos.
- Repite el proceso con cada grupo hasta que cada subgrupo sea "puro" (todos de la misma clase) o hasta que se cumpla algun criterio de parada.

Vamos a verlo con un ejemplo tan pequeno que podemos seguirlo a mano.


## 1.2 Ejemplo juguete: aprobar o reprobar un examen

Imaginemos 8 estudiantes. Sabemos cuantas horas estudiaron y si asistieron a clase. Queremos predecir si aprobaron o no.


In [ ]:
# Creamos un dataset diminuto a mano
datos_estudiantes = pd.DataFrame({
    "horas_estudio":  [1, 2, 3, 4, 5, 6, 7, 8],
    "asistio_clase":  [0, 0, 1, 0, 1, 1, 1, 1],   # 0 = No, 1 = Si
    "aprobo":         [0, 0, 0, 0, 1, 1, 1, 1],    # 0 = Reprobo, 1 = Aprobo
})

print("Nuestros datos de juguete:")
print(datos_estudiantes.to_string(index=False))
print(f"\nTotal: {len(datos_estudiantes)} estudiantes")
print(f"Aprobaron: {datos_estudiantes['aprobo'].sum()}")
print(f"Reprobaron: {(datos_estudiantes['aprobo'] == 0).sum()}")

## 1.3 Entrenemos nuestro primer arbol

Usamos `DecisionTreeClassifier` de scikit-learn. Es literalmente tres lineas de codigo.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Separamos las variables de entrada (X) y la variable objetivo (y)
X = datos_estudiantes[["horas_estudio", "asistio_clase"]]
y = datos_estudiantes["aprobo"]

# Creamos el arbol y lo entrenamos
arbol = DecisionTreeClassifier(random_state=SEED)
arbol.fit(X, y)

print("Arbol entrenado.")
print(f"Profundidad del arbol: {arbol.get_depth()}")
print(f"Numero de hojas: {arbol.get_n_leaves()}")

## 1.4 Visualicemos el arbol

Esta es la gran ventaja de los arboles: podemos *ver* exactamente que decidieron. Cada caja nos dice:

- **La pregunta** que hace (por ejemplo, "horas_estudio <= 4.5").
- **Gini**: una medida de "impureza" (0 = perfecto, 0.5 = mezcla total).
- **Samples**: cuantos datos llegaron a ese nodo.
- **Value**: cuantos de cada clase hay ahi.
- **Class**: la clase mayoritaria en ese nodo.


In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(12, 6))

plot_tree(
    arbol,
    feature_names=["horas_estudio", "asistio_clase"],
    class_names=["Reprobo", "Aprobo"],
    filled=True,           # Colorea los nodos segun la clase mayoritaria
    rounded=True,          # Bordes redondeados
    fontsize=11,
    ax=ax
)

ax.set_title("Nuestro primer arbol de decision", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

**Como leer el arbol:**

Empezamos por el nodo de arriba (la raiz). El arbol pregunta, por ejemplo, "horas_estudio <= 4.5?". Si la respuesta es **si**, vamos a la izquierda. Si es **no**, vamos a la derecha. Repetimos hasta llegar a una hoja (nodo final), que nos da la prediccion.

Esto es exactamente lo que hace el modelo cuando le pasamos datos nuevos.


## 1.5 Hagamos predicciones a mano

Vamos a pasar un estudiante nuevo y seguir el camino que recorre dentro del arbol.


In [ ]:
# Un estudiante nuevo: 5 horas de estudio, si asistio a clase
estudiante_nuevo = pd.DataFrame({
    "horas_estudio": [5],
    "asistio_clase": [1]
})

prediccion = arbol.predict(estudiante_nuevo)
probabilidad = arbol.predict_proba(estudiante_nuevo)

print("Estudiante nuevo:")
print(f"  Horas de estudio: 5")
print(f"  Asistio a clase:  Si")
print(f"\nPrediccion: {'Aprobo' if prediccion[0] == 1 else 'Reprobo'}")
print(f"Probabilidad de reprobar: {probabilidad[0][0]:.0%}")
print(f"Probabilidad de aprobar:  {probabilidad[0][1]:.0%}")

## 1.6 La importancia de las variables (Feature Importance)

El arbol nos dice cuales variables fueron mas utiles para tomar decisiones. Esto es invaluable en la practica: nos ayuda a entender nuestros datos.


In [ ]:
importancias = pd.Series(
    arbol.feature_importances_,
    index=["horas_estudio", "asistio_clase"]
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 3))
importancias.plot(kind="barh", color=["#0891B2", "#F97316"], ax=ax, edgecolor="white")
ax.set_xlabel("Importancia")
ax.set_title("Importancia de cada variable en el arbol", fontweight="bold")

# Anotamos los valores
for i, (val, name) in enumerate(zip(importancias.values, importancias.index)):
    ax.text(val + 0.01, i, f"{val:.2f}", va="center", fontsize=12)

plt.tight_layout()
plt.show()

print("Interpretacion:")
print("Un valor mas alto significa que esa variable fue mas util para separar las clases.")

## 1.7 El problema del overfitting

Un arbol sin restricciones puede volverse muy profundo y "memorizar" cada dato de entrenamiento, incluyendo el ruido. Esto se llama **overfitting** y es el mayor enemigo de los arboles de decision.

Veamoslo con un ejemplo visual: creamos datos con algo de ruido y comparamos un arbol sin limites contra uno con profundidad controlada.


In [ ]:
from sklearn.datasets import make_moons

# Creamos datos con forma de lunas (un clasico para visualizar clasificacion)
X_lunas, y_lunas = make_moons(n_samples=200, noise=0.3, random_state=SEED)

# Dos arboles: uno sin limites y otro con profundidad maxima de 3
arbol_libre = DecisionTreeClassifier(random_state=SEED)
arbol_limitado = DecisionTreeClassifier(max_depth=3, random_state=SEED)

arbol_libre.fit(X_lunas, y_lunas)
arbol_limitado.fit(X_lunas, y_lunas)

print(f"Arbol SIN limites  -> profundidad: {arbol_libre.get_depth()}, hojas: {arbol_libre.get_n_leaves()}")
print(f"Arbol CON limites  -> profundidad: {arbol_limitado.get_depth()}, hojas: {arbol_limitado.get_n_leaves()}")

In [ ]:
# Funcion auxiliar para graficar las fronteras de decision
def graficar_frontera(modelo, X, y, ax, titulo):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdYlBu")
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#EF4444", edgecolors="white", s=40, label="Clase 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#0891B2", edgecolors="white", s=40, label="Clase 1")
    ax.set_title(titulo, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

graficar_frontera(arbol_libre, X_lunas, y_lunas, axes[0],
                  f"Sin limites (profundidad={arbol_libre.get_depth()})")
graficar_frontera(arbol_limitado, X_lunas, y_lunas, axes[1],
                  f"Con max_depth=3 (profundidad={arbol_limitado.get_depth()})")

fig.suptitle("Overfitting vs. modelo controlado", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Observa como el arbol sin limites (izquierda) crea fronteras")
print("muy irregulares tratando de clasificar CADA punto correctamente.")
print("El arbol limitado (derecha) es mas suave y generalizara mejor con datos nuevos.")

---
---
# PARTE 2: Random Forest

## 2.1 La idea: muchas opiniones son mejor que una

Un solo arbol puede equivocarse facilmente (especialmente si hace overfitting). Pero, si entrenamos **muchos arboles diferentes** y los hacemos **votar**, los errores individuales tienden a cancelarse.

Esto se llama **ensamble** (ensemble), y Random Forest es la version mas conocida.

**Como funciona Random Forest:**
1. Toma muestras aleatorias de los datos (con reemplazo, es decir, bootstrap).
2. Entrena un arbol de decision con cada muestra.
3. En cada nodo del arbol, solo considera un subconjunto aleatorio de variables.
4. Para predecir, todos los arboles votan y gana la clase con mas votos.

Esa doble aleatoridad (datos + variables) hace que los arboles sean diferentes entre si, lo cual es clave para que el ensamble funcione.


## 2.2 Ejemplo juguete: viendo la votacion en accion

Volvamos a nuestros datos de estudiantes y entrenemos un Random Forest de solo 5 arboles para poder ver que hace cada uno.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Un bosque pequeno: 5 arboles
bosque_mini = RandomForestClassifier(
    n_estimators=5,         # Solo 5 arboles para poder inspeccionarlos
    max_depth=3,
    random_state=SEED
)

# Usamos los datos de estudiantes
X = datos_estudiantes[["horas_estudio", "asistio_clase"]]
y = datos_estudiantes["aprobo"]

bosque_mini.fit(X, y)

# Veamos que predice cada arbol individual para nuestro estudiante nuevo
estudiante_nuevo = pd.DataFrame({"horas_estudio": [5], "asistio_clase": [1]})

print("Prediccion de CADA arbol individual:")
print("-" * 45)
votos = []
for i, arbol_i in enumerate(bosque_mini.estimators_):
    pred = arbol_i.predict(estudiante_nuevo)[0]
    clase = "Aprobo" if pred == 1 else "Reprobo"
    votos.append(pred)
    print(f"  Arbol {i+1}: {clase}")

print("-" * 45)
print(f"  Votos por 'Aprobo':  {sum(votos)} de {len(votos)}")
print(f"  Votos por 'Reprobo': {len(votos) - sum(votos)} de {len(votos)}")
print(f"\nResultado final (por mayoria): {'Aprobo' if sum(votos) > len(votos)/2 else 'Reprobo'}")

## 2.3 Visualicemos los 5 arboles del bosque

Cada arbol es ligeramente diferente porque recibio datos y variables distintas.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for i, arbol_i in enumerate(bosque_mini.estimators_):
    plot_tree(
        arbol_i,
        feature_names=["horas_estudio", "asistio_clase"],
        class_names=["Reprobo", "Aprobo"],
        filled=True,
        rounded=True,
        fontsize=7,
        ax=axes[i]
    )
    axes[i].set_title(f"Arbol {i+1}", fontweight="bold", fontsize=11)

fig.suptitle("Los 5 arboles de nuestro mini-bosque (cada uno es diferente)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Nota como cada arbol tiene una estructura diferente.")
print("Eso es intencional: la diversidad es lo que hace fuerte al bosque.")

## 2.4 Efecto del numero de arboles

Una pregunta natural es: cuantos arboles necesito? Veamoslo graficamente con los datos de lunas.


In [ ]:
from sklearn.model_selection import cross_val_score

# Probamos diferentes cantidades de arboles
n_arboles_lista = [1, 3, 5, 10, 25, 50, 100, 200, 500]
scores = []

for n in n_arboles_lista:
    rf = RandomForestClassifier(n_estimators=n, random_state=SEED)
    # Usamos validacion cruzada con 5 folds para medir bien el desempeno
    score = cross_val_score(rf, X_lunas, y_lunas, cv=5, scoring="accuracy").mean()
    scores.append(score)
    print(f"  {n:>3} arboles -> accuracy promedio: {score:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_arboles_lista, scores, marker="o", color="#0891B2", linewidth=2, markersize=8)
ax.fill_between(n_arboles_lista, scores, alpha=0.1, color="#0891B2")
ax.set_xlabel("Numero de arboles (n_estimators)")
ax.set_ylabel("Accuracy (validacion cruzada)")
ax.set_title("Efecto del numero de arboles en Random Forest", fontweight="bold")
ax.set_xscale("log")  # Escala logaritmica para ver mejor
ax.set_xticks(n_arboles_lista)
ax.set_xticklabels(n_arboles_lista)

# Marcamos el mejor
mejor_idx = np.argmax(scores)
ax.annotate(f"Mejor: {n_arboles_lista[mejor_idx]} arboles\n({scores[mejor_idx]:.4f})",
            xy=(n_arboles_lista[mejor_idx], scores[mejor_idx]),
            xytext=(n_arboles_lista[mejor_idx]*1.5, scores[mejor_idx]-0.02),
            arrowprops=dict(arrowstyle="->", color="#F97316"),
            fontsize=11, color="#F97316", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nObservaciones:")
print("- El mayor salto ocurre al pasar de 1 a unos pocos arboles.")
print("- Despues de cierto punto (50-100), la mejora es minima.")
print("- Mas arboles = mas tiempo de entrenamiento sin ganancia significativa.")

## 2.5 Random Forest vs arbol simple: fronteras de decision

Comparemos visualmente como se ven las fronteras de decision.


In [ ]:
rf_completo = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_completo.fit(X_lunas, y_lunas)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

graficar_frontera(arbol_libre, X_lunas, y_lunas, axes[0],
                  "1 arbol (sin limites)")
graficar_frontera(rf_completo, X_lunas, y_lunas, axes[1],
                  "Random Forest (100 arboles)")

fig.suptitle("Un arbol solo vs. Random Forest", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("El Random Forest (derecha) produce fronteras mas suaves y robustas.")
print("Los errores individuales de los arboles se cancelan entre si.")

---
---
# PARTE 3: Gradient Boosting y XGBoost

## 3.1 La idea: aprender de los errores

Random Forest entrena muchos arboles **en paralelo** (independientes entre si). Gradient Boosting toma un enfoque diferente: entrena arboles **en serie**, donde cada arbol nuevo se enfoca en corregir los errores del arbol anterior.

Es como un estudiante que despues de cada examen revisa sus errores y estudia especificamente esos temas para el siguiente examen.

**El proceso paso a paso:**
1. Entrenar un primer arbol simple.
2. Calcular los errores (residuos) de ese arbol.
3. Entrenar un segundo arbol para predecir *esos errores*.
4. Sumar las predicciones del primer arbol + la correccion del segundo.
5. Repetir: cada arbol nuevo corrige los errores acumulados.


## 3.2 Visualicemos el proceso de correccion

Vamos a usar datos simples y mostrar como cada arbol va reduciendo el error.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Entrenamos modelos con diferentes cantidades de arboles para ver la progresion
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
etapas = [1, 3, 10, 50]

for idx, n in enumerate(etapas):
    gb = GradientBoostingClassifier(
        n_estimators=n,
        max_depth=2,
        learning_rate=0.3,
        random_state=SEED
    )
    gb.fit(X_lunas, y_lunas)
    acc = gb.score(X_lunas, y_lunas)
    graficar_frontera(gb, X_lunas, y_lunas, axes[idx],
                      f"{n} arbol{'es' if n > 1 else ''} (acc={acc:.2f})")

fig.suptitle("Gradient Boosting: como las fronteras mejoran con cada arbol",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Con solo 1 arbol la frontera es muy simple (underfitting).")
print("A medida que anadimos arboles, las correcciones van refinando la frontera.")
print("Con 50 arboles ya captura bien la forma de los datos.")

## 3.3 La curva de error: viendo la mejora en tiempo real

Gradient Boosting nos permite ver como el error va bajando con cada arbol anadido. Esto es muy util para decidir cuantos arboles usar.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Dividimos los datos de lunas en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_lunas, y_lunas, test_size=0.3, random_state=SEED
)

# Entrenamos con 200 arboles
gb_200 = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.1,
    random_state=SEED
)
gb_200.fit(X_train, y_train)

# Calculamos el accuracy en cada etapa (despues de cada arbol anadido)
errores_train = []
errores_test = []

for i, y_pred_train in enumerate(gb_200.staged_predict(X_train)):
    errores_train.append(1 - accuracy_score(y_train, y_pred_train))

for i, y_pred_test in enumerate(gb_200.staged_predict(X_test)):
    errores_test.append(1 - accuracy_score(y_test, y_pred_test))

fig, ax = plt.subplots(figsize=(10, 5))
etapas_range = range(1, len(errores_train) + 1)
ax.plot(etapas_range, errores_train, label="Error en entrenamiento", color="#0891B2", linewidth=2)
ax.plot(etapas_range, errores_test, label="Error en prueba (test)", color="#F97316", linewidth=2)
ax.set_xlabel("Numero de arboles")
ax.set_ylabel("Tasa de error")
ax.set_title("Curva de error de Gradient Boosting", fontweight="bold")
ax.legend(fontsize=12)

# Linea vertical en el mejor punto de test
mejor_test = np.argmin(errores_test)
ax.axvline(mejor_test + 1, color="#EF4444", linestyle="--", alpha=0.7)
ax.annotate(f"Mejor en test:\n{mejor_test + 1} arboles",
            xy=(mejor_test + 1, errores_test[mejor_test]),
            xytext=(mejor_test + 40, errores_test[mejor_test] + 0.05),
            arrowprops=dict(arrowstyle="->", color="#EF4444"),
            fontsize=11, color="#EF4444")

plt.tight_layout()
plt.show()

print(f"Mejor error en test: {errores_test[mejor_test]:.4f} (con {mejor_test + 1} arboles)")
print("\nNota: si usamos demasiados arboles, el error de entrenamiento sigue")
print("bajando pero el de test puede empezar a subir. Eso es overfitting.")

## 3.4 XGBoost: la version turbo

XGBoost (eXtreme Gradient Boosting) es una implementacion optimizada de Gradient Boosting que agrega:

- **Regularizacion** incorporada (para evitar overfitting).
- **Velocidad** (implementado en C++ con paralelizacion).
- **Manejo automatico de datos faltantes** (no hay que rellenar los NaN).

Es el algoritmo que ha ganado la mayoria de competencias de Kaggle en datos tabulares.

Usarlo es tan simple como los demas modelos de scikit-learn:


In [ ]:
from xgboost import XGBClassifier

# Creamos y entrenamos el modelo
xgb_modelo = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=SEED,
    eval_metric="logloss"     # Metrica para clasificacion binaria
)

xgb_modelo.fit(X_train, y_train)

# Evaluamos
acc_train = xgb_modelo.score(X_train, y_train)
acc_test = xgb_modelo.score(X_test, y_test)

print(f"Accuracy en entrenamiento: {acc_train:.4f}")
print(f"Accuracy en test:          {acc_test:.4f}")

## 3.5 Comparacion visual: Random Forest vs Gradient Boosting vs XGBoost

Veamos las tres fronteras de decision lado a lado.


In [ ]:
rf_comp = RandomForestClassifier(n_estimators=100, random_state=SEED)
gb_comp = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=SEED)
xgb_comp = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=SEED, eval_metric="logloss")

rf_comp.fit(X_lunas, y_lunas)
gb_comp.fit(X_lunas, y_lunas)
xgb_comp.fit(X_lunas, y_lunas)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

modelos = [rf_comp, gb_comp, xgb_comp]
nombres = ["Random Forest", "Gradient Boosting", "XGBoost"]
colores_titulo = ["#10B981", "#F97316", "#8B5CF6"]

for ax, modelo, nombre, color in zip(axes, modelos, nombres, colores_titulo):
    graficar_frontera(modelo, X_lunas, y_lunas, ax, nombre)
    ax.title.set_color(color)

fig.suptitle("Comparacion de fronteras de decision", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Los tres producen fronteras diferentes:")
print("- Random Forest: fronteras con bordes 'escalonados' (por la votacion).")
print("- Gradient Boosting: fronteras mas suaves, ajustadas progresivamente.")
print("- XGBoost: similar a GB pero con regularizacion que suaviza aun mas.")

---
---
# PARTE 4: Comparacion practica con datos reales

## 4.1 Cargando un dataset real

Vamos a usar el dataset de **cancer de mama de Wisconsin** (breast cancer), que viene incluido en scikit-learn. Lo convertiremos a un DataFrame de pandas, exactamente como harias si cargaras un CSV o un Excel.

El objetivo es clasificar tumores como **benignos** (no peligrosos) o **malignos** (cancerosos) basandose en mediciones de las celulas.


In [ ]:
from sklearn.datasets import load_breast_cancer

# Cargamos el dataset
cancer_data = load_breast_cancer()

# Lo convertimos a DataFrame de pandas (igual que si leyeramos un CSV)
df = pd.DataFrame(
    data=cancer_data.data,
    columns=cancer_data.feature_names
)
df["diagnostico"] = cancer_data.target   # 0 = maligno, 1 = benigno

print(f"Dimensiones: {df.shape[0]} pacientes, {df.shape[1]} columnas")
print(f"\nPrimeras 5 filas:")
df.head()

**Nota practica:** Si tuvieras tus propios datos, los cargarias asi:

```python
# Desde un CSV
df = pd.read_csv("mi_archivo.csv")

# Desde un Excel
df = pd.read_excel("mi_archivo.xlsx")
```

A partir de aqui, todo el codigo seria identico.


In [ ]:
# Veamos la distribucion de clases
conteo = df["diagnostico"].value_counts()
etiquetas = {0: "Maligno", 1: "Benigno"}

print("Distribucion de diagnosticos:")
for clase, cantidad in conteo.items():
    print(f"  {etiquetas[clase]}: {cantidad} ({cantidad/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
barras = ax.bar(
    [etiquetas[0], etiquetas[1]],
    [conteo[0], conteo[1]],
    color=["#EF4444", "#10B981"],
    edgecolor="white",
    width=0.5
)

for barra, val in zip(barras, [conteo[0], conteo[1]]):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 5,
            str(val), ha="center", fontweight="bold", fontsize=13)

ax.set_ylabel("Cantidad de pacientes")
ax.set_title("Distribucion de diagnosticos", fontweight="bold")
plt.tight_layout()
plt.show()

## 4.2 Preparacion de los datos

Separamos en variables de entrada (X) y variable objetivo (y), y luego en conjuntos de entrenamiento y prueba.


In [ ]:
# Variables de entrada y objetivo
X = df.drop("diagnostico", axis=1)
y = df["diagnostico"]

# Division 70% entrenamiento, 30% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} pacientes")
print(f"Datos de prueba:        {X_test.shape[0]} pacientes")
print(f"\nVariables disponibles:  {X_train.shape[1]}")
print(f"\nAlgunas de las variables:")
for col in X.columns[:10]:
    print(f"  - {col}")
print(f"  ... y {X.shape[1] - 10} mas")

## 4.3 Entrenamiento de los tres modelos

Ahora entrenamos Regresion Logistica (como baseline), Random Forest y XGBoost.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Definimos los tres modelos
modelos = {
    "Regresion Logistica": LogisticRegression(max_iter=10000, random_state=SEED),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=SEED),
    "XGBoost":             XGBClassifier(n_estimators=100, learning_rate=0.1,
                                         random_state=SEED, eval_metric="logloss"),
}

# Entrenamos y evaluamos cada uno
resultados = []

for nombre, modelo in modelos.items():
    # Entrenar
    modelo.fit(X_train, y_train)

    # Predecir
    y_pred = modelo.predict(X_test)

    # Calcular metricas
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    resultados.append({
        "Modelo": nombre,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })

    print(f"{nombre}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print()

df_resultados = pd.DataFrame(resultados)

## 4.4 Comparacion visual de los modelos

Vamos a graficar las cuatro metricas para los tres modelos, una al lado de la otra.


In [ ]:
metricas = ["Accuracy", "Precision", "Recall", "F1-Score"]
colores_modelos = ["#0891B2", "#10B981", "#F97316"]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metricas))
ancho = 0.22

for i, (_, fila) in enumerate(df_resultados.iterrows()):
    valores = [fila[m] for m in metricas]
    barras = ax.bar(x + i * ancho, valores, ancho,
                    label=fila["Modelo"], color=colores_modelos[i],
                    edgecolor="white", linewidth=0.5)
    # Anotamos los valores
    for barra, val in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.005,
                f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")

ax.set_xticks(x + ancho)
ax.set_xticklabels(metricas, fontsize=12)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel("Score")
ax.set_title("Comparacion de modelos en el dataset de cancer de mama", fontweight="bold", fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

## 4.5 Matriz de confusion

La matriz de confusion nos dice exactamente *donde* se equivoca cada modelo. Es especialmente importante en medicina: no es lo mismo fallar diciendo que alguien esta sano cuando esta enfermo (falso negativo) que al reves.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
nombres_modelos = list(modelos.keys())
colores_cm = ["Blues", "Greens", "Oranges"]

for idx, (nombre, modelo) in enumerate(modelos.items()):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(cm, display_labels=["Maligno", "Benigno"])
    disp.plot(ax=axes[idx], cmap=colores_cm[idx], colorbar=False)
    axes[idx].set_title(nombre, fontweight="bold", fontsize=13)

fig.suptitle("Matrices de confusion", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Lectura rapida de la matriz:")
print("  - Diagonal principal (arriba-izq y abajo-der) = predicciones CORRECTAS")
print("  - Fuera de la diagonal = ERRORES")
print("  - Arriba-derecha = Falsos positivos (dijo benigno, era maligno)")
print("  - Abajo-izquierda = Falsos negativos (dijo maligno, era benigno)")

## 4.6 Importancia de variables (Feature Importance)

Veamos cuales variables son las mas importantes para Random Forest y XGBoost. Esto nos ayuda a entender *que mide el modelo* para tomar sus decisiones.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, (nombre, modelo, color) in enumerate([
    ("Random Forest", modelos["Random Forest"], "#10B981"),
    ("XGBoost", modelos["XGBoost"], "#F97316")
]):
    importancias = pd.Series(
        modelo.feature_importances_,
        index=X.columns
    ).sort_values(ascending=True)

    # Mostramos solo las top 15
    top_15 = importancias.tail(15)
    top_15.plot(kind="barh", ax=axes[idx], color=color, edgecolor="white")
    axes[idx].set_title(f"Top 15 variables - {nombre}", fontweight="bold")
    axes[idx].set_xlabel("Importancia")

fig.suptitle("Variables mas importantes para cada modelo", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Observaciones:")
print("- Ambos modelos coinciden en varias de las variables mas importantes.")
print("- Las variables relacionadas con el 'peor caso' (worst_*) tienden a ser las mas relevantes.")
print("- Esto tiene sentido clinico: los valores extremos de las celulas")
print("  son buenos indicadores de malignidad.")

---
---
# PARTE 5: Hiperparametros y busqueda automatica

## 5.1 Que son los hiperparametros?

Los **hiperparametros** son las configuraciones que nosotros decidimos *antes* de entrenar un modelo. El modelo no los aprende de los datos; somos nosotros quienes los elegimos.

Es como una receta de cocina: los ingredientes son los datos, el algoritmo es la receta, pero la temperatura del horno y el tiempo de coccion son los hiperparametros.

Algunos hiperparametros importantes:

| Modelo | Hiperparametro | Que controla |
|--------|---------------|-------------|
| Arbol de Decision | `max_depth` | Profundidad maxima del arbol |
| Arbol de Decision | `min_samples_split` | Minimo de datos para dividir un nodo |
| Random Forest | `n_estimators` | Numero de arboles en el bosque |
| Random Forest | `max_features` | Variables consideradas en cada split |
| XGBoost | `learning_rate` | Cuanto peso tiene cada arbol nuevo |
| XGBoost | `n_estimators` | Numero de arboles |
| XGBoost | `subsample` | Fraccion de datos usada por arbol |


## 5.2 Impacto de un hiperparametro: ejemplo visual

Antes de buscar automaticamente, veamos *a mano* como cambia el rendimiento al variar un solo hiperparametro: `max_depth` en un arbol de decision.


In [ ]:
profundidades = list(range(1, 16))
scores_train = []
scores_test = []

for d in profundidades:
    arbol_d = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    arbol_d.fit(X_train, y_train)
    scores_train.append(arbol_d.score(X_train, y_train))
    scores_test.append(arbol_d.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(profundidades, scores_train, marker="o", label="Entrenamiento", color="#0891B2", linewidth=2)
ax.plot(profundidades, scores_test, marker="s", label="Prueba (test)", color="#F97316", linewidth=2)
ax.set_xlabel("max_depth (profundidad maxima)")
ax.set_ylabel("Accuracy")
ax.set_title("Efecto de max_depth en el arbol de decision", fontweight="bold")
ax.legend(fontsize=12)
ax.set_xticks(profundidades)

# Marcamos el punto optimo en test
mejor_d = profundidades[np.argmax(scores_test)]
ax.axvline(mejor_d, color="#10B981", linestyle="--", alpha=0.7, label=f"Mejor: max_depth={mejor_d}")
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nMejor profundidad en test: {mejor_d} (accuracy: {max(scores_test):.4f})")
print("\nObserva como el accuracy de entrenamiento sigue subiendo (el modelo memoriza)")
print("pero el de test se estabiliza o baja. Ese es el punto de overfitting.")

## 5.3 GridSearchCV: busqueda exhaustiva

GridSearchCV prueba **todas las combinaciones** posibles de hiperparametros que le indicamos. Es como probar todos los platos del menu.

Usemoslo con Random Forest.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Definimos la "grilla" de hiperparametros a probar
parametros_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],     # None = sin limite
    "min_samples_split": [2, 5, 10],
}

# Calculamos cuantas combinaciones son
n_combinaciones = 1
for v in parametros_grid.values():
    n_combinaciones *= len(v)
print(f"Total de combinaciones a probar: {n_combinaciones}")
print(f"Con 5-fold cross-validation: {n_combinaciones * 5} modelos a entrenar")
print(f"\nEsto puede tardar un momento...\n")

# Creamos el buscador
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=SEED),
    param_grid=parametros_grid,
    cv=5,                       # Validacion cruzada con 5 folds
    scoring="accuracy",
    n_jobs=-1,                  # Usar todos los nucleos disponibles
    verbose=0
)

# Ejecutamos la busqueda
grid_search.fit(X_train, y_train)

print("Busqueda completada.")
print(f"\nMejores hiperparametros encontrados:")
for param, valor in grid_search.best_params_.items():
    print(f"  {param}: {valor}")
print(f"\nMejor accuracy (cross-validation): {grid_search.best_score_:.4f}")
print(f"Accuracy en test:                  {grid_search.score(X_test, y_test):.4f}")

## 5.4 Visualicemos los resultados del GridSearch

Es util ver como las diferentes combinaciones de hiperparametros afectan el rendimiento.


In [ ]:
# Extraemos los resultados
resultados_grid = pd.DataFrame(grid_search.cv_results_)

# Graficamos accuracy vs max_depth, agrupado por n_estimators
fig, ax = plt.subplots(figsize=(10, 5))

for n_est in [50, 100, 200]:
    mascara = resultados_grid["param_n_estimators"] == n_est
    datos_filtrados = resultados_grid[mascara].copy()
    # Agrupamos por max_depth (promediando sobre min_samples_split)
    grouped = datos_filtrados.groupby("param_max_depth")["mean_test_score"].mean()
    etiquetas = [str(x) if x is not None else "Sin limite" for x in grouped.index]
    ax.plot(etiquetas, grouped.values, marker="o", linewidth=2, label=f"{n_est} arboles")

ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy promedio (CV)")
ax.set_title("Resultados del GridSearch: accuracy por configuracion", fontweight="bold")
ax.legend(fontsize=11, title="n_estimators")

plt.tight_layout()
plt.show()

print("Este grafico nos permite ver rapidamente que combinaciones funcionan mejor.")

## 5.5 RandomizedSearchCV: busqueda aleatoria

Cuando hay muchos hiperparametros, probar todas las combinaciones es impracticable. RandomizedSearchCV prueba un numero fijo de combinaciones **al azar**. Suele encontrar soluciones casi tan buenas en una fraccion del tiempo.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Para RandomizedSearch podemos usar distribuciones continuas
parametros_random = {
    "n_estimators": randint(50, 500),          # Entero entre 50 y 500
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": randint(2, 20),       # Entero entre 2 y 20
    "min_samples_leaf": randint(1, 10),        # Entero entre 1 y 10
    "max_features": ["sqrt", "log2", None],
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=SEED),
    param_distributions=parametros_random,
    n_iter=50,                  # Solo 50 combinaciones aleatorias
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=SEED,
    verbose=0
)

print("Probando 50 combinaciones aleatorias (en vez de miles)...\n")
random_search.fit(X_train, y_train)

print("Busqueda completada.")
print(f"\nMejores hiperparametros encontrados:")
for param, valor in random_search.best_params_.items():
    print(f"  {param}: {valor}")
print(f"\nMejor accuracy (cross-validation): {random_search.best_score_:.4f}")
print(f"Accuracy en test:                  {random_search.score(X_test, y_test):.4f}")

## 5.6 Comparacion: GridSearch vs RandomizedSearch vs XGBoost con tuning

Hagamos tambien un tuning de XGBoost y comparemos todos los resultados finales.


In [ ]:
# Tuning de XGBoost con RandomizedSearch
parametros_xgb = {
    "n_estimators": randint(50, 300),
    "max_depth": randint(2, 10),
    "learning_rate": uniform(0.01, 0.3),     # Continuo entre 0.01 y 0.31
    "subsample": uniform(0.6, 0.4),          # Continuo entre 0.6 y 1.0
    "colsample_bytree": uniform(0.6, 0.4),
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=SEED, eval_metric="logloss"),
    param_distributions=parametros_xgb,
    n_iter=50,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=SEED,
    verbose=0
)

print("Buscando mejores hiperparametros para XGBoost...\n")
xgb_search.fit(X_train, y_train)

print("Mejores hiperparametros XGBoost:")
for param, valor in xgb_search.best_params_.items():
    if isinstance(valor, float):
        print(f"  {param}: {valor:.4f}")
    else:
        print(f"  {param}: {valor}")

In [ ]:
# Comparacion final de todos los modelos
modelos_finales = {
    "Logistica (baseline)": LogisticRegression(max_iter=10000, random_state=SEED),
    "RF (sin tuning)": RandomForestClassifier(n_estimators=100, random_state=SEED),
    "RF (GridSearch)": grid_search.best_estimator_,
    "RF (RandomSearch)": random_search.best_estimator_,
    "XGB (sin tuning)": XGBClassifier(n_estimators=100, random_state=SEED, eval_metric="logloss"),
    "XGB (RandomSearch)": xgb_search.best_estimator_,
}

resultados_finales = []

for nombre, modelo in modelos_finales.items():
    # Los que ya fueron entrenados por *Search no necesitan re-fit
    if "Search" not in nombre and "baseline" not in nombre:
        modelo.fit(X_train, y_train)
    elif "baseline" in nombre:
        modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    resultados_finales.append({"Modelo": nombre, "Accuracy": acc, "F1-Score": f1})

df_final = pd.DataFrame(resultados_finales).sort_values("Accuracy", ascending=False)
print("Resultados finales (ordenados por Accuracy):")
print(df_final.to_string(index=False))

In [ ]:
# Grafico de barras final
fig, ax = plt.subplots(figsize=(12, 6))

colores = ["#8B5CF6", "#10B981", "#10B981", "#10B981", "#F97316", "#F97316"]
y_pos = range(len(df_final))

barras = ax.barh(
    y_pos,
    df_final["Accuracy"].values,
    color=[colores[i] for i in range(len(df_final))],
    edgecolor="white",
    height=0.6
)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_final["Modelo"].values, fontsize=11)
ax.set_xlabel("Accuracy", fontsize=12)
ax.set_title("Comparacion final de todos los modelos", fontweight="bold", fontsize=14)
ax.set_xlim(0.90, 1.005)

# Anotamos los valores
for barra, val in zip(barras, df_final["Accuracy"].values):
    ax.text(val + 0.001, barra.get_y() + barra.get_height()/2,
            f"{val:.4f}", va="center", fontsize=11, fontweight="bold")

# Leyenda de colores
leyenda = [
    mpatches.Patch(color="#8B5CF6", label="Reg. Logistica"),
    mpatches.Patch(color="#10B981", label="Random Forest"),
    mpatches.Patch(color="#F97316", label="XGBoost"),
]
ax.legend(handles=leyenda, loc="lower right", fontsize=10)

plt.tight_layout()
plt.show()

print("\nConclusiones:")
print("1. Incluso sin tuning, RF y XGBoost superan a la regresion logistica.")
print("2. El tuning de hiperparametros puede mejorar el rendimiento, aunque")
print("   en este dataset la mejora es modesta porque los datos ya son 'faciles'.")
print("3. RandomizedSearch logra resultados comparables a GridSearch en menos tiempo.")

---
---
# Resumen final

## Lo que aprendimos

| Tema | Concepto clave |
|------|---------------|
| Arbol de Decision | Divide datos con preguntas si/no. Simple pero propenso a overfitting. |
| Random Forest | Muchos arboles votan en paralelo. Robusto y facil de usar. |
| Gradient Boosting | Arboles en serie, cada uno corrige al anterior. Mas preciso pero mas lento. |
| XGBoost | Version optimizada de Gradient Boosting. Gana competencias de ML. |
| GridSearchCV | Prueba todas las combinaciones. Exhaustivo pero lento. |
| RandomizedSearchCV | Prueba combinaciones al azar. Rapido y casi tan bueno. |

## Regla de oro

1. **Empieza simple**: entrena una regresion logistica como baseline.
2. **Sube de nivel**: prueba Random Forest (poca configuracion, buenos resultados).
3. **Si necesitas mas**: usa XGBoost con tuning de hiperparametros.
4. **Siempre compara** con metricas objetivas (accuracy, precision, recall, F1).
5. **Cuidado con el overfitting**: usa validacion cruzada para evaluar de verdad.

## Para seguir practicando

- Prueba estos modelos con tus propios datos (CSV o Excel).
- Experimenta cambiando los hiperparametros y observa el efecto.
- Intenta con otros datasets de sklearn: `load_iris()`, `load_wine()`, `load_digits()`.
- Investiga `LightGBM` y `CatBoost`, otras variantes modernas de Gradient Boosting.


---
*Notebook creado para el Diplomado en Machine Learning - Curso III, Bloque 1*
